# ML3 — Recommendation functions

**Project:** DS 5020 Mini-Project "Customers Like You".

Input: the frozen `data/processed/A_log.npz` and `product_index.parquet` (read-only). Code lives in `src/recommend.py`
(unit-tested in `tests/test_recommend.py`). **No evaluation here** — `test_targets.parquet` is never read; that is ML4.

What is built:

1. `TruncatedSVD(k=50)` — k is provisional from ML2 and is a parameter.
2. Product vectors `P` (one unit-length row per product), with two weightings left available for ML4 to choose:
   `"V"` = `components_.T` (brief's starter code) and `"VSigma"` = `components_.T * singular_values_`.
3. `similar_products(j, n)` — cosine neighbours of a product, excluding itself.
4. Nightly **"Customers who bought this also bought"** table for every product (batched, no dense 3687×3687).
5. `recommend_for_customers(rows, n)` — scores = `A_log[rows] @ P @ P.T`, computed through the k-dim space; already-bought
   products masked to −∞.
6. `best_seller_recs(rows, n)` — baseline: train best sellers by distinct customers, minus what the customer bought.

Customers are referred to **only by anonymised row index**, never by Customer ID.

In [1]:
# Environment guard: this notebook must run in the "giftlane" conda env (pandas 3.0.6).
# It fails fast with "Wrong Python environment. Run: conda activate giftlane" otherwise.
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.env_check import check_env

print("environment OK:", check_env())

environment OK: /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python3.11


## 0. Setup and load the frozen matrix

In [2]:
import time

import numpy as np
import pandas as pd
import sklearn

from src.svd import load_inputs
from src.recommend import (
    assert_no_bought_in_recs, best_seller_order, best_seller_recs, customer_top_bought, describe_cols,
    find_col_by_description, fit_recommender, precompute_item_neighbors, product_vectors,
    recommend_for_customers, save_product_factors, similar_products, with_weighting,
)

ROOT = _ROOT
PROCESSED = ROOT / "data" / "processed"
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.max_columns", None)

A, product_index, customer_index = load_inputs(ROOT)
n_customers, n_products = A.shape
print("python  :", sys.executable)
print("sklearn :", sklearn.__version__, "| numpy :", np.__version__, "| pandas :", pd.__version__)
print(f"A_log   : {n_customers:,} x {n_products:,}, nnz {A.nnz:,}")

python  : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
sklearn : 1.9.1 | numpy : 2.4.6 | pandas : 3.0.6
A_log   : 5,471 x 3,687, nnz 413,706


## 1–2. Fit SVD (k = 50) and build product vectors

Both weightings are constructed from the **same** fitted SVD. `"V"` is the default used in the rest of this notebook;
`"VSigma"` is built alongside so ML4 can compare them. Rows are L2-normalised with `sklearn.preprocessing.normalize`,
so `P @ P.T` is cosine similarity; zero-norm rows (none expected after the min-customers filter) would stay zero.

In [3]:
K = 50
rec = fit_recommender(A, product_index, k=K, weighting="V")
rec_vs = with_weighting(rec, "VSigma")  # same SVD object, different scaling

print(f"SVD fit (k={K}, randomized, n_iter=7, seed=42): {rec.fit_seconds:.3f}s")
print(f"P ('V')      shape {rec.P.shape}, dtype {rec.P.dtype}, row norms min/max = "
      f"{np.linalg.norm(rec.P, axis=1).min():.6f} / {np.linalg.norm(rec.P, axis=1).max():.6f}, zero-norm rows: {len(rec.zero_norm_cols)}")
print(f"P ('VSigma') shape {rec_vs.P.shape}, row norms min/max = "
      f"{np.linalg.norm(rec_vs.P, axis=1).min():.6f} / {np.linalg.norm(rec_vs.P, axis=1).max():.6f}, zero-norm rows: {len(rec_vs.zero_norm_cols)}")
cos_between = np.sum(rec.P * rec_vs.P, axis=1)
print(f"cosine between a product's 'V' and 'VSigma' vector: median {np.median(cos_between):.3f}, min {cos_between.min():.3f}  "
      f"(they differ, so the choice matters -> ML4)")
assert rec_vs.svd is rec.svd

SVD fit (k=50, randomized, n_iter=7, seed=42): 0.107s
P ('V')      shape (3687, 50), dtype float64, row norms min/max = 1.000000 / 1.000000, zero-norm rows: 0
P ('VSigma') shape (3687, 50), row norms min/max = 1.000000 / 1.000000, zero-norm rows: 0
cosine between a product's 'V' and 'VSigma' vector: median 0.885, min 0.752  (they differ, so the choice matters -> ML4)


## 3. `similar_products` sanity checks

Top 10 cosine neighbours for four well-known products and one low-popularity product (10–15 train customers).

In [4]:
names = [
    "WHITE HANGING HEART T-LIGHT HOLDER",
    "REGENCY CAKESTAND 3 TIER",
    "LUNCH BAG RED RETROSPOT",
    "PARTY BUNTING",
]
cols = {name: find_col_by_description(rec, name) for name in names}

low = product_index.loc[product_index["n_customers"].between(10, 15)].sort_values("StockCode")
low_col = int(low["col"].iloc[0])
cols[f"{low['Description'].iloc[0]}  (low popularity, {int(low['n_customers'].iloc[0])} customers)"] = low_col
print(f"{len(low)} products have 10-15 train customers; using the first by StockCode as the low-popularity example")

for label, j in cols.items():
    idx, sc = similar_products(rec, j, n=10)
    assert j not in idx
    pi = product_index.iloc[j]
    print(f"\n=== {label}  [col {j}, StockCode {pi['StockCode']}, {int(pi['n_customers'])} train customers] ===")
    print(describe_cols(rec, idx, sc).to_string())

393 products have 10-15 train customers; using the first by StockCode as the low-popularity example

=== WHITE HANGING HEART T-LIGHT HOLDER  [col 3432, StockCode 85123A, 1422 train customers] ===
       col StockCode                         Description  n_customers_train     score
rank                                                                                 
1      910     21733    RED HANGING HEART T-LIGHT HOLDER                691  0.894949
2     1842     22804     CANDLEHOLDER PINK HANGING HEART                321  0.767560
3      909     21732  RED HANGING METAL STAR TLIGHT HLDR                 50  0.589727
4     2723     71477   COLOUR GLASS. STAR T-LIGHT HOLDER                282  0.537943
5      344     20994          JAZZ HEARTS SPIRAL NOTEPAD                 12  0.451062
6     1267     22189             CREAM HEART CARD HOLDER                497  0.446643
7     2856     79532      MOSAIC GLASS BALL CANDLEHOLDER                 15  0.409331
8     2714     71053          

## 4. Nightly item-neighbour table

Top 10 similar products for **every** product, computed in 512-row batches of `P[batch] @ P.T` (each batch is
512×3687 float64 ≈ 15 MB; the full 3687×3687 matrix is never built). Saved to `data/processed/item_neighbors.parquet`.

In [5]:
neighbors, secs_nb = precompute_item_neighbors(rec, n=10, batch_size=512)
print(f"precompute: {secs_nb:.3f}s for {n_products:,} products x 10 neighbours = {len(neighbors):,} rows")
print(f"(for reference, a full dense {n_products}x{n_products} similarity matrix would be "
      f"{n_products**2 * 4 / 1e6:.1f} MB in float32 / {n_products**2 * 8 / 1e6:.1f} MB in float64 - not built)")
assert len(neighbors) == n_products * 10
assert not (neighbors["product_col"] == neighbors["neighbor_col"]).any(), "a product is its own neighbour"
assert neighbors.groupby("product_col")["rank"].apply(lambda s: s.tolist() == list(range(1, 11))).all()

nb_path = PROCESSED / "item_neighbors.parquet"
neighbors.to_parquet(nb_path, index=False)
print(f"saved -> {nb_path.relative_to(ROOT)} ({nb_path.stat().st_size / 1e6:.2f} MB)")
print(neighbors.dtypes.to_string())
display(neighbors.head(10))
print("\nneighbour score distribution (rank 1 / rank 10):")
print(neighbors.loc[neighbors['rank'] == 1, 'score'].describe().round(3).to_string())
print(neighbors.loc[neighbors['rank'] == 10, 'score'].describe().round(3).to_string())

precompute: 0.097s for 3,687 products x 10 neighbours = 36,870 rows
(for reference, a full dense 3687x3687 similarity matrix would be 54.4 MB in float32 / 108.8 MB in float64 - not built)
saved -> data/processed/item_neighbors.parquet (0.67 MB)
product_col               int64
StockCode                   str
Description                 str
rank                      int64
neighbor_col              int64
neighbor_StockCode          str
neighbor_Description        str
score                   float64


,product_col,StockCode,Description,rank,neighbor_col,neighbor_StockCode,neighbor_Description,score
0,0,10002,INFLATABLE POLITICAL GLOBE,1,176,20704,MR ROBOT SOFT TOY,0.517756
1,0,10002,INFLATABLE POLITICAL GLOBE,2,908,21731,RED TOADSTOOL LED NIGHT LIGHT,0.510594
2,0,10002,INFLATABLE POLITICAL GLOBE,3,954,21789,KIDS RAIN MAC PINK,0.482749
3,0,10002,INFLATABLE POLITICAL GLOBE,4,1259,22181,SNOWSTORM PHOTO FRAME FRIDGE MAGNET,0.460058
4,0,10002,INFLATABLE POLITICAL GLOBE,5,1365,22299,PIG KEYRING WITH LIGHT & SOUND,0.459899
5,0,10002,INFLATABLE POLITICAL GLOBE,6,3144,84796B,BLUE SAVANNAH PICNIC HAMPER FOR 2,0.454567
6,0,10002,INFLATABLE POLITICAL GLOBE,7,683,21441,BLUE BIRDHOUSE DECORATION,0.451337
7,0,10002,INFLATABLE POLITICAL GLOBE,8,2841,79320,FLAMINGO LIGHTS,0.451033
8,0,10002,INFLATABLE POLITICAL GLOBE,9,3477,85164B,BLACK BAROQUE CUCKOO CLOCK,0.448369
9,0,10002,INFLATABLE POLITICAL GLOBE,10,887,21708,FOLDING UMBRELLA CREAM/MULTI SPOT,0.444448



neighbour score distribution (rank 1 / rank 10):
count    3687.000
mean        0.811
std         0.115
min         0.457
25%         0.733
50%         0.827
75%         0.905
max         0.999
count    3687.000
mean        0.651
std         0.135
min         0.317
25%         0.545
50%         0.634
75%         0.747
max         0.988


## 5. `recommend_for_customers` for all customers

`scores = (A_log[rows] @ P) @ P.T` in 512-customer batches: each batch is (512×k) then (512×3687). Products the
customer already bought are set to −∞ before the top 10 is taken. Timed over all 5,471 rows.

In [6]:
all_rows = np.arange(n_customers)
recs, rec_scores, secs_rec = recommend_for_customers(rec, all_rows, n=10, batch_size=512)
print(f"scored + ranked {n_customers:,} customers x {n_products:,} products in {secs_rec:.3f}s  "
      f"({secs_rec / n_customers * 1e3:.3f} ms per customer)")
print(f"recs shape {recs.shape}, dtype {recs.dtype}")
assert_no_bought_in_recs(rec, all_rows, recs)
print("assert OK: no recommendation is a product the customer already bought (all 5,471 customers)")
assert (np.sort(recs, axis=1)[:, 1:] != np.sort(recs, axis=1)[:, :-1]).all(), "duplicate recs within a customer"
print("assert OK: no duplicate products within a customer's top 10")

nightly = rec.fit_seconds + secs_nb + secs_rec
print(f"\nnightly job estimate (fit {rec.fit_seconds:.2f}s + item neighbours {secs_nb:.2f}s + "
      f"score all customers {secs_rec:.2f}s) = {nightly:.2f}s   [J1 target: < 60 s]")

scored + ranked 5,471 customers x 3,687 products in 0.161s  (0.029 ms per customer)
recs shape (5471, 10), dtype int64
assert OK: no recommendation is a product the customer already bought (all 5,471 customers)
assert OK: no duplicate products within a customer's top 10

nightly job estimate (fit 0.11s + item neighbours 0.10s + score all customers 0.16s) = 0.37s   [J1 target: < 60 s]


## 6. Best-seller baseline

Train best sellers by distinct customers (`product_index.n_customers`), skipping products each customer already
bought. The global top 10 is shown first; per-customer lists differ only by the exclusions.

In [7]:
order = best_seller_order(rec)
print("global top-10 best sellers in TRAIN (by distinct customers):")
print(describe_cols(rec, order[:10]).to_string())

bs_recs, secs_bs = best_seller_recs(rec, all_rows, n=10)
assert_no_bought_in_recs(rec, all_rows, bs_recs)
print(f"\nbaseline for {n_customers:,} customers in {secs_bs:.3f}s; assert OK: no already-bought product recommended")
overlap = np.array([len(set(a) & set(b)) for a, b in zip(recs, bs_recs)])
print(f"overlap between SVD top-10 and best-seller top-10 per customer: mean {overlap.mean():.2f} products, "
      f"share of customers with zero overlap {np.mean(overlap == 0):.1%}")

global top-10 best sellers in TRAIN (by distinct customers):
       col StockCode                         Description  n_customers_train
rank                                                                       
1     3432    85123A  WHITE HANGING HEART T-LIGHT HOLDER               1422
2     1478     22423            REGENCY CAKESTAND 3 TIER               1213
3      501     21212    PACK OF 72 RETRO SPOT CAKE CASES               1058
4     1219     22138       BAKING SET 9 PIECE RETROSPOT                 975
5     3211     84879       ASSORTED COLOUR BIRD ORNAMENT                907
6     3411    85099B             JUMBO BAG RED RETROSPOT                898
7     2653     47566                       PARTY BUNTING                855
8     1220     22139    RETROSPOT TEA SET CERAMIC 11 PC                 836
9      189     20725             LUNCH BAG RED RETROSPOT                805
10    1511     22457     NATURAL SLATE HEART CHALKBOARD                 796



baseline for 5,471 customers in 0.124s; assert OK: no already-bought product recommended
overlap between SVD top-10 and best-seller top-10 per customer: mean 0.02 products, share of customers with zero overlap 97.6%


## 7. Three customers by anonymised row index

For rows 0, 1000 and 3000: the 5 products with the largest `A_log` value (most bought), the SVD top 10, and the
best-seller top 10. Only the row index is shown.

In [8]:
for row in (0, 1000, 3000):
    r = rec.A.getrow(row)
    print(f"\n################ customer row {row}  ({r.nnz} distinct products bought in train) ################")
    print("--- 5 most-bought products ---")
    print(customer_top_bought(rec, row, n=5).to_string())
    print("--- SVD top 10 recommendations ---")
    print(describe_cols(rec, recs[row], rec_scores[row]).to_string())
    print("--- best-seller baseline top 10 ---")
    print(describe_cols(rec, bs_recs[row]).to_string())
    assert set(recs[row]).isdisjoint(r.indices) and set(bs_recs[row]).isdisjoint(r.indices)


################ customer row 0  (25 distinct products bought in train) ################
--- 5 most-bought products ---
       col StockCode                     Description  n_customers_train  A_log_value  approx_total_qty
rank                                                                                                  
1     2167     23166  MEDIUM CERAMIC TOP STORAGE JAR                107    11.214735             74215
2       20   15056BL         EDWARDIAN PARASOL BLACK                301     0.693147                 1
3       21    15056N       EDWARDIAN PARASOL NATURAL                331     0.693147                 1
4       22    15056P          EDWARDIAN PARASOL PINK                175     0.693147                 1
5      157     20679           EDWARDIAN PARASOL RED                252     0.693147                 1
--- SVD top 10 recommendations ---
       col StockCode                          Description  n_customers_train      score
rank                               

## 8. Save product vectors

`P` for the default weighting `"V"` at k = 50, float64. (`"VSigma"` can be rebuilt from the same SVD at any time.)

In [9]:
p_path = save_product_factors(rec, ROOT)
print(f"saved -> {p_path.relative_to(ROOT)}: shape {rec.P.shape}, dtype {rec.P.dtype}, "
      f"{p_path.stat().st_size / 1e6:.2f} MB on disk ({rec.P.nbytes / 1e6:.2f} MB in memory)")
U = rec.A @ rec.svd.components_.T  # customer factors would add this much if stored
print(f"for reference: customer factors (n x k float64) would be {U.nbytes / 1e6:.2f} MB; "
      f"both together {(U.nbytes + rec.P.nbytes) / 1e6:.2f} MB  [J1 target: factors < 5 MB; dense A_log 161.4 MB]")

saved -> data/processed/product_factors_k50.npy: shape (3687, 50), dtype float64, 1.47 MB on disk (1.47 MB in memory)
for reference: customer factors (n x k float64) would be 2.19 MB; both together 3.66 MB  [J1 target: factors < 5 MB; dense A_log 161.4 MB]


## Notes

- Both product-vector weightings (`"V"`, `"VSigma"`) are available; **nothing here chooses between them** — ML4 does,
  on the held-out targets.
- k = 50 is provisional from ML2 and is a parameter of `fit_recommender`.
- No test data was read in this notebook.